# Lab Day 1 — Xây dựng mạng nơ-ron và thí nghiệm huấn luyện

**Sinh viên:** Nguyễn Văn Điền | **MSSV:** 02615 | **Track 4 · Ngày 1 · VinUniversity AICB 2026**

> Notebook này hoàn thiện toàn bộ Part 0 đến Part 4, phủ đầy đủ 7 chủ đề thí nghiệm, đo độ nhiễu seed, đánh giá mô hình cuối cùng trên eval và xuất báo cáo/bảng biểu.

In [ ]:
# ===== Cấu hình đường dẫn, thiết bị và imports =====
import os, sys, json, time, subprocess
import numpy as np, torch
from pathlib import Path

# Tự động nhận diện cấu trúc thư mục (chạy từ submission_02615/code, repo code/, hoặc Colab)
if os.path.exists('../../data'):
    REPO_ROOT = '../..'
    OUT_DIR = '..'
elif os.path.exists('../data'):
    REPO_ROOT = '..'
    OUT_DIR = '../submission_02615'
elif os.path.exists('data'):
    REPO_ROOT = '.'
    OUT_DIR = 'submission_02615'
else:
    REPO_ROOT = '/content/K4-Track4-Day1-Neuralnetwork-02615-NguyenVanDien'
    OUT_DIR = f'{REPO_ROOT}/submission_02615'

os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/results', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'BF16 Supported: {torch.cuda.is_bf16_supported()}')

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, build_scheduler, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Chuẩn bị dữ liệu
Chia sẵn bằng metadata (`data/split_metadata.csv`): chạy `scripts/split_data.py` (tạo `data/processed/train.npz` và `eval.npz`).
Tách validation 20% từ train (phân tầng theo nhãn, seed 42). Chuẩn hoá 10 đặc trưng liên tục CHỈ dựa trên train.

In [ ]:
# 1. Chia dữ liệu nếu chưa có
processed_dir = os.path.join(REPO_ROOT, 'data', 'processed')
if not os.path.exists(os.path.join(processed_dir, 'train.npz')):
    print('Chạy scripts/split_data.py...')
    subprocess.run([sys.executable, os.path.join(REPO_ROOT, 'scripts', 'split_data.py')], cwd=REPO_ROOT, check=True)

# 2. Nạp dữ liệu và chuẩn hoá trên train
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

# 3. Kiểm tra tính chuẩn hoá trên 10 cột số liên tục
X_tr_np = data['X_tr'].cpu().numpy()
means = np.mean(X_tr_np[:, :10], axis=0)
stds = np.std(X_tr_np[:, :10], axis=0)
print('Kiểm tra 10 đặc trưng số trên Train sau chuẩn hoá:')
print('  Max mean lệch khỏi 0:', float(np.max(np.abs(means))))
print('  Max std lệch khỏi 1: ', float(np.max(np.abs(stds - 1.0))))


## Part 1 — Định nghĩa model và kiểm tra 'sức khoẻ' ban đầu
- `M-base`: `54 → 256 → 128 → 7`, đúng **47 879 tham số**, logits `(B, 7)`, không softmax trong model.
- Thực hiện 3 phép thử sức khoẻ: (1) Shape, (2) Loss bước 0 $\approx \ln 7 \approx 1.946$, (3) Quá khớp 20 mẫu về 0, (4) Kiểm tra gradient chảy tới mọi tham số.

In [ ]:
# 1. Khởi tạo model và assert số tham số
model_sanity = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
p_count = count_params(model_sanity)
assert p_count == EXPECTED_PARAMS[(256, 128)], f'Sai số tham số: {p_count} != 47879'
print(f'✓ Model M-base tạo thành công với đúng {p_count} tham số (khớp 47 879)!')

# 2. Forward kiểm tra shape
x_dummy = torch.randn(8, 54, device=device)
logits_dummy = model_sanity(x_dummy)
assert logits_dummy.shape == (8, 7), f'Logits shape lỗi: {logits_dummy.shape}'
print(f'✓ Forward lô (8, 54) -> shape logits: {tuple(logits_dummy.shape)}')

# 3. Phép thử 1: Loss bước 0 trên Val
step0_val = evaluate(model_sanity, data['X_val'], data['y_val'], loss_name='ce')
expected_ln7 = float(np.log(7))
print(f'✓ Loss bước 0 trên Val: {step0_val["loss"]:.4f} (Lý thuyết ln(7) = {expected_ln7:.4f}, sai lệch = {abs(step0_val["loss"] - expected_ln7):.4f})')

# 4. Phép thử 2: Quá khớp 20 mẫu về 0
x_tiny = data['X_tr'][:20]
y_tiny = data['y_tr'][:20]
tiny_model = MLP(hidden=(256, 128), dropout=0.0, init='he').to(device)
tiny_opt = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
for step in range(300):
    tiny_opt.zero_grad()
    loss_tiny = torch.nn.functional.cross_entropy(tiny_model(x_tiny), y_tiny)
    loss_tiny.backward()
    tiny_opt.step()
acc_tiny = (torch.argmax(tiny_model(x_tiny), dim=1) == y_tiny).float().mean().item()
print(f'✓ Quá khớp 20 mẫu sau 300 bước: Loss = {loss_tiny.item():.6f}, Accuracy = {acc_tiny*100:.1f}%')

# 5. Phép thử 3: Gradient chảy tới mọi tham số
tiny_opt.zero_grad()
loss_check = torch.nn.functional.cross_entropy(tiny_model(x_tiny), y_tiny)
loss_check.backward()
print('✓ Kiểm tra gradient của từng lớp:')
for name, param in tiny_model.named_parameters():
    assert param.grad is not None, f'Grad None tại {name}'
    gn = torch.norm(param.grad).item()
    assert gn > 0.0, f'Grad = 0 tại {name}'
    print(f'  {name:20s}: shape {tuple(param.shape):15s} | grad_norm = {gn:.6f}')


**Nhận xét Part 1:**
- Model `M-base` được khởi tạo chuẩn xác với đúng 47 879 tham số.
- Loss bước 0 đo được trên tập validation sát với $\ln 7 \approx 1.9459$, chứng tỏ trọng số ban đầu được khởi tạo cân bằng giữa 7 lớp, không bị thiên lệch.
- Mô hình quá khớp hoàn toàn 20 mẫu về loss gần bằng 0 và accuracy 100%, khẳng định kiến trúc mạng và vòng lặp backward hoạt động chính xác.
- Tất cả các lớp đều có gradient khác 0 sau backward, không xảy ra hiện tượng triệt tiêu gradient (vanishing gradient).

## Part 2 — Pipeline và Baseline (Đo độ nhiễu seed)
Baseline: `M-base`, He init, CE loss, SGD+momentum 0.9, lr=0.05, batch 512, 20 epochs.
Chạy baseline với **3 seed khác nhau (1, 2, 3)** để đo độ nhiễu seed (trung bình $\mu$ và độ lệch chuẩn $\sigma$), thiết lập ngưỡng nhiễu $2\sigma$.

In [ ]:
baseline_seeds = [1, 2, 3]
baseline_results = []

for s in baseline_seeds:
    cfg_base = {
        **DEFAULT_CFG,
        'exp_id': f'base-s{s}',
        'seed': s,
        'lr': 0.05,
        'description': f'Baseline M-base (seed {s})'
    }
    print(f'\n>>> Huấn luyện Baseline {cfg_base["exp_id"]}...')
    res = run_experiment(cfg_base, data)
    save_result(res, f'{OUT_DIR}/results')
    plot_run(res, f'{OUT_DIR}/figures/{cfg_base["exp_id"]}.png')
    baseline_results.append(res)

val_f1s = [r['summary']['val_macro_f1'] for r in baseline_results]
val_accs = [r['summary']['val_acc'] for r in baseline_results]
mean_f1, std_f1 = float(np.mean(val_f1s)), float(np.std(val_f1s))
mean_acc, std_acc = float(np.mean(val_accs)), float(np.std(val_accs))
noise_2sigma = 2 * std_f1

print('\n' + '='*50)
print('TỔNG HỢP BASELINE (3 SEEDS):')
print(f'  Val Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}')
print(f'  Val Accuracy: {mean_acc:.4f} ± {std_acc:.4f}')
print(f'  Ngưỡng nhiễu 2σ (Macro-F1): {noise_2sigma:.4f}')
print('='*50)


**Nhận xét Baseline:**
- Baseline vượt xa mốc "đoán luôn lớp đa số" (0.4876), đạt val accuracy và macro-F1 ổn định.
- Độ nhiễu giữa các seed cho phép ta xác định ngưỡng $2\sigma$. Mọi kết luận so sánh ở Part 3 chỉ được xem là có ý nghĩa thống kê khi mức chênh lệch $> 2\sigma$.

## Part 3 — 7 Chủ đề thí nghiệm
Thực hiện đầy đủ 7 chủ đề thí nghiệm: (1) Hàm mất mát, (2) Bộ tối ưu hoá, (3) Hyper-parameters, (4) Dropout, (5) Gradient clipping, (6) Mixed precision, (7) Khởi tạo tham số.
Mỗi thí nghiệm tuân thủ nghiêm ngặt nguyên tắc chỉ thay đổi 1 yếu tố so với baseline.

### Chủ đề 1: Hàm mất mát (Loss function) — Cross-Entropy vs MSE

In [ ]:
# Dự đoán: Cross-Entropy cho gradient lớn hơn khi dự đoán sai lệch nhiều, giúp hội tụ nhanh hơn MSE.
cfg_mse = {
    **DEFAULT_CFG,
    'exp_id': 'loss-mse',
    'group': 'loss',
    'loss': 'mse',
    'description': 'Hàm mất mát MSE trên nhãn one-hot'
}
res_mse = run_experiment(cfg_mse, data)
save_result(res_mse, f'{OUT_DIR}/results')
plot_run(res_mse, f'{OUT_DIR}/figures/{cfg_mse["exp_id"]}.png')
plot_compare([baseline_results[0], res_mse], 'val_macro_f1', f'{OUT_DIR}/figures/compare_loss.png', 'So sánh CE vs MSE (Val Macro-F1)')


### Chủ đề 2: Bộ tối ưu hoá (Optimizers: SGD, SGD+Momentum, Adam, AdamW)

In [ ]:
# Thử nghiệm mỗi bộ tối ưu ở nhiều learning rate để tìm cấu hình tối ưu của từng bộ
opt_configs = [
    {'exp_id': 'opt-sgd-lr0.05', 'group': 'optimizer', 'optimizer': 'sgd', 'lr': 0.05, 'description': 'SGD thuần lr=0.05'},
    {'exp_id': 'opt-sgd-lr0.1', 'group': 'optimizer', 'optimizer': 'sgd', 'lr': 0.1, 'description': 'SGD thuần lr=0.1'},
    {'exp_id': 'opt-sgdm-lr0.02', 'group': 'optimizer', 'optimizer': 'sgd_momentum', 'lr': 0.02, 'description': 'SGD+Momentum lr=0.02'},
    {'exp_id': 'opt-adam-lr1e-3', 'group': 'optimizer', 'optimizer': 'adam', 'lr': 0.001, 'description': 'Adam lr=1e-3'},
    {'exp_id': 'opt-adam-lr3e-4', 'group': 'optimizer', 'optimizer': 'adam', 'lr': 0.0003, 'description': 'Adam lr=3e-4'},
    {'exp_id': 'opt-adamw-lr1e-3', 'group': 'optimizer', 'optimizer': 'adamw', 'lr': 0.001, 'weight_decay': 0.01, 'description': 'AdamW lr=1e-3, wd=0.01'},
    {'exp_id': 'opt-adamw-lr3e-4', 'group': 'optimizer', 'optimizer': 'adamw', 'lr': 0.0003, 'weight_decay': 0.01, 'description': 'AdamW lr=3e-4, wd=0.01'},
]
opt_results = [baseline_results[0]]
for c in opt_configs:
    cfg_exp = {**DEFAULT_CFG, **c}
    print(f'>>> Chạy {cfg_exp["exp_id"]}...')
    r = run_experiment(cfg_exp, data)
    save_result(r, f'{OUT_DIR}/results')
    plot_run(r, f'{OUT_DIR}/figures/{cfg_exp["exp_id"]}.png')
    opt_results.append(r)

plot_compare(opt_results, 'val_macro_f1', f'{OUT_DIR}/figures/compare_optimizer.png', 'So sánh các bộ tối ưu (Val Macro-F1)')


### Chủ đề 3: Hyper-parameters (Batch Size & Dung lượng mạng M-wide, M-deep)

In [ ]:
hparam_configs = [
    {'exp_id': 'hparam-batch128', 'group': 'hparam', 'batch': 128, 'description': 'Batch size nhỏ 128 (nhiều bước cập nhật hơn)'},
    {'exp_id': 'hparam-batch2048', 'group': 'hparam', 'batch': 2048, 'description': 'Batch size lớn 2048 (ít bước cập nhật hơn)'},
    {'exp_id': 'hparam-mwide', 'group': 'hparam', 'hidden': (512, 256), 'description': 'Mô hình rộng M-wide (161k params)'},
    {'exp_id': 'hparam-mdeep', 'group': 'hparam', 'hidden': (256, 128, 64), 'description': 'Mô hình sâu M-deep (55k params)'},
]
hparam_results = [baseline_results[0]]
for c in hparam_configs:
    cfg_exp = {**DEFAULT_CFG, **c}
    print(f'>>> Chạy {cfg_exp["exp_id"]}...')
    r = run_experiment(cfg_exp, data)
    save_result(r, f'{OUT_DIR}/results')
    plot_run(r, f'{OUT_DIR}/figures/{cfg_exp["exp_id"]}.png')
    hparam_results.append(r)

plot_compare(hparam_results, 'val_macro_f1', f'{OUT_DIR}/figures/compare_hparam.png', 'So sánh Hyper-parameters (Val Macro-F1)')


### Chủ đề 4: Dropout (q = 0.2, 0.5)

In [ ]:
dropout_configs = [
    {'exp_id': 'drop-0.2', 'group': 'dropout', 'dropout': 0.2, 'description': 'Dropout q=0.2 sau ReLU lớp ẩn'},
    {'exp_id': 'drop-0.5', 'group': 'dropout', 'dropout': 0.5, 'description': 'Dropout q=0.5 sau ReLU lớp ẩn'},
]
dropout_results = [baseline_results[0]]
for c in dropout_configs:
    cfg_exp = {**DEFAULT_CFG, **c}
    print(f'>>> Chạy {cfg_exp["exp_id"]}...')
    r = run_experiment(cfg_exp, data)
    save_result(r, f'{OUT_DIR}/results')
    plot_run(r, f'{OUT_DIR}/figures/{cfg_exp["exp_id"]}.png')
    dropout_results.append(r)

plot_compare(dropout_results, 'val_loss', f'{OUT_DIR}/figures/compare_dropout.png', 'So sánh Dropout (Val Loss)')


### Chủ đề 5: Gradient Clipping (Cắt gradient)

In [ ]:
clip_configs = [
    {'exp_id': 'clip-1.0-normallr', 'group': 'clipping', 'clip_norm': 1.0, 'description': 'Clip max_norm=1.0 ở lr thường 0.05'},
    {'exp_id': 'noclip-highlr', 'group': 'clipping', 'lr': 0.8, 'clip_norm': None, 'description': 'Không clip ở lr cao 0.8'},
    {'exp_id': 'clip-1.0-highlr', 'group': 'clipping', 'lr': 0.8, 'clip_norm': 1.0, 'description': 'Clip max_norm=1.0 ở lr cao 0.8'},
]
clip_results = [baseline_results[0]]
for c in clip_configs:
    cfg_exp = {**DEFAULT_CFG, **c}
    print(f'>>> Chạy {cfg_exp["exp_id"]}...')
    r = run_experiment(cfg_exp, data)
    save_result(r, f'{OUT_DIR}/results')
    plot_run(r, f'{OUT_DIR}/figures/{cfg_exp["exp_id"]}.png')
    clip_results.append(r)

plot_compare(clip_results, 'grad_norm', f'{OUT_DIR}/figures/compare_clipping.png', 'So sánh Gradient Norm khi Clip và Không Clip')


### Chủ đề 6: Mixed Precision (AMP FP16 / BF16)

In [ ]:
amp_configs = [
    {'exp_id': 'amp-fp16', 'group': 'amp', 'precision': 'fp16', 'description': 'Mixed Precision FP16 với GradScaler'},
]
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    amp_configs.append({'exp_id': 'amp-bf16', 'group': 'amp', 'precision': 'bf16', 'description': 'Mixed Precision BF16'})

amp_results = [baseline_results[0]]
for c in amp_configs:
    cfg_exp = {**DEFAULT_CFG, **c}
    print(f'>>> Chạy {cfg_exp["exp_id"]}...')
    r = run_experiment(cfg_exp, data)
    save_result(r, f'{OUT_DIR}/results')
    plot_run(r, f'{OUT_DIR}/figures/{cfg_exp["exp_id"]}.png')
    amp_results.append(r)

plot_compare(amp_results, 'val_macro_f1', f'{OUT_DIR}/figures/compare_amp.png', 'So sánh độ chính xác FP32 vs Mixed Precision')


### Chủ đề 7: Khởi tạo tham số (Weight Initialization: He, Xavier, Normal, Zeros)

In [ ]:
init_configs = [
    {'exp_id': 'init-xavier', 'group': 'init', 'init': 'xavier', 'description': 'Khởi tạo Xavier Normal'},
    {'exp_id': 'init-normal', 'group': 'init', 'init': 'normal', 'description': 'Khởi tạo Normal std=0.01'},
    {'exp_id': 'init-zeros', 'group': 'init', 'init': 'zeros', 'description': 'Khởi tạo toàn bộ W=0 (đối xứng)'},
]
init_results = [baseline_results[0]]
for c in init_configs:
    cfg_exp = {**DEFAULT_CFG, **c}
    print(f'>>> Chạy {cfg_exp["exp_id"]}...')
    # Thống kê độ lệch chuẩn kích hoạt bước 0
    m_temp = MLP(hidden=cfg_exp['hidden'], dropout=cfg_exp['dropout'], init=cfg_exp['init']).to(device)
    act_stds = activation_stats(m_temp, data['X_val'][:512])
    print(f'  Activation stds sau các lớp ({cfg_exp["init"]}) ở bước 0:', [round(s, 5) for s in act_stds])
    r = run_experiment(cfg_exp, data)
    save_result(r, f'{OUT_DIR}/results')
    plot_run(r, f'{OUT_DIR}/figures/{cfg_exp["exp_id"]}.png')
    init_results.append(r)

plot_compare(init_results, 'val_macro_f1', f'{OUT_DIR}/figures/compare_init.png', 'So sánh các phương pháp khởi tạo tham số')


## Part 4 — Đánh giá cuối trên tập Eval, Lập bảng Excel và Đóng gói bài nộp
- Lựa chọn cấu hình tốt nhất thuần tuý dựa trên tập **Validation** (không dùng eval).
- Dự đoán trên toàn bộ `eval.npz` (116 203 dòng) và xuất file `predictions_eval.csv`.
- Chạy `scripts/evaluate.py` để tính điểm chính thức, ghi `eval_result.json` và phân tích lỗi theo lớp.
- Xuất toàn bộ kết quả vào file `experiments.xlsx`.

In [ ]:
# 1. Tự động tìm cấu hình có Val Macro-F1 cao nhất từ tất cả thí nghiệm hợp lệ
all_results = load_results(f'{OUT_DIR}/results')
valid_runs = [r for r in all_results if not r['summary'].get('diverged', False)]
best_run = max(valid_runs, key=lambda r: r['summary']['val_macro_f1'])
best_cfg = best_run['cfg']
print(f'Cấu hình tốt nhất theo Validation: {best_cfg["exp_id"]} ({best_cfg["description"]})')
print(f'Val Macro-F1 tốt nhất: {best_run["summary"]["val_macro_f1"]:.4f}')

# 2. Chạy cấu hình cuối cùng (Final Best Model) với Cosine LR Scheduler để tối ưu hoá thêm
final_cfg = {
    **best_cfg,
    'exp_id': 'final-best',
    'group': 'final',
    'scheduler': 'cosine',
    'description': 'Cấu hình tối ưu cuối cùng (Best Val + Cosine LR Scheduler)'
}
print('\n>>> Huấn luyện Cấu hình cuối cùng (final-best)...')
res_final = run_experiment(final_cfg, data)
save_result(res_final, f'{OUT_DIR}/results')
plot_run(res_final, f'{OUT_DIR}/figures/{final_cfg["exp_id"]}.png')

# 3. Dự đoán trên tập Eval và chấm điểm bằng scripts/evaluate.py
pred_path = f'{OUT_DIR}/predictions_eval.csv'
eval_json_path = f'{OUT_DIR}/eval_result.json'
final_eval(final_cfg, res_final, data, pred_path)

# Chạy script evaluate chính thức
eval_cmd = [
    sys.executable, os.path.join(REPO_ROOT, 'scripts', 'evaluate.py'),
    '--pred', pred_path,
    '--data', os.path.join(REPO_ROOT, 'data', 'covtype.csv.gz'),
    '--meta', os.path.join(REPO_ROOT, 'data', 'split_metadata.csv'),
    '--out', eval_json_path
]
subprocess.run(eval_cmd, check=True)

with open(eval_json_path, 'r', encoding='utf-8') as fp:
    eval_result = json.load(fp)
print(f'\nOfficial Eval Accuracy: {eval_result["accuracy"]:.4f}')
print(f'Official Eval Macro-F1: {eval_result["macro_f1"]:.4f}')


In [ ]:
# Điền kết quả vào file experiments.xlsx từ template
all_results = load_results(f'{OUT_DIR}/results')
rows = []
for r in all_results:
    exp_id = r['cfg']['exp_id']
    eval_scores = None
    if exp_id in ('base-s1', 'final-best'):
        eval_scores = eval_result if exp_id == 'final-best' else None
    row = to_row(r, eval_scores=eval_scores)
    rows.append(row)

template_path = os.path.join(REPO_ROOT, 'templates', 'experiment_table_template.xlsx')
out_xlsx_path = os.path.join(OUT_DIR, 'experiments.xlsx')
write_xlsx(rows, template_path, out_xlsx_path)
print(f'✓ Đã hoàn tất ghi {len(rows)} thí nghiệm vào {out_xlsx_path}!')
